# Rotation calibration — curve fit

Data: `rotation_sweep.csv`, collected by `tests/calibrate/rotation_sweep.py`.
Each row is one **raw** (gain=1) in-place spin: a commanded angle `cmd_deg`
(+ = CCW) at wheel speed `speed_mms`, with the camera-measured `actual_deg`
and `error_deg = actual − cmd`.

Goal: find the simplest model `error = f(cmd, speed)` that predicts the turn
error, so the turn command can pre-compensate. A single gain is `error ≈ b·cmd`;
we test whether speed and a quadratic term help.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

CSV = Path('rotation_sweep.csv')
d = np.genfromtxt(CSV, delimiter=',', names=True)
cmd, speed = d['cmd_deg'], d['speed_mms']
actual, err = d['actual_deg'], d['error_deg']
print(f'{len(cmd)} rows | speeds {sorted(set(speed.tolist()))} | angles {sorted(set(np.abs(cmd).tolist()))}')

In [ ]:
# Error vs commanded angle, coloured by speed
fig, ax = plt.subplots(figsize=(9, 5))
for s in sorted(set(speed.tolist())):
    m = speed == s
    ax.scatter(cmd[m], err[m], label=f'{int(s)} mm/s')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('commanded angle (deg, + = CCW)')
ax.set_ylabel('error = actual − cmd (deg)')
ax.set_title('rotation error vs commanded angle, by speed')
ax.legend(); ax.grid(alpha=0.3)

In [ ]:
# Fit candidate polynomial models for error = f(cmd, speed)
def fit(cols, names):
    X = np.column_stack(cols)
    coef, *_ = np.linalg.lstsq(X, err, rcond=None)
    pred = X @ coef
    res = err - pred
    ss_res = float((res ** 2).sum())
    ss_tot = float(((err - err.mean()) ** 2).sum())
    r2 = 1 - ss_res / ss_tot if ss_tot else 0.0
    n, p = len(err), X.shape[1]
    adj = 1 - (1 - r2) * (n - 1) / (n - p - 1) if n - p - 1 > 0 else float('nan')
    rmse = (ss_res / n) ** 0.5
    print(f"{'  +  '.join(names):34s}  R2={r2:.3f}  adjR2={adj:.3f}  RMSE={rmse:5.2f} deg")
    return dict(names=names, coef=coef, r2=r2, rmse=rmse, pred=pred)

o = np.ones_like(cmd)
a, s = cmd, speed
print('model                                fit')
m1 = fit([o, a],                         ['1', 'a'])
m2 = fit([o, a, s],                      ['1', 'a', 's'])
m3 = fit([o, a, s, a * s],               ['1', 'a', 's', 'a*s'])
m4 = fit([o, a, s, a * s, a * a],        ['1', 'a', 's', 'a*s', 'a^2'])
m5 = fit([o, a, s, a * s, a * a, s * s], ['1', 'a', 's', 'a*s', 'a^2', 's^2'])

In [ ]:
# Pick the model after eyeballing the table above (simplest with good adjR2/RMSE)
best = m4
print('chosen model:', '  +  '.join(best['names']))
for nm, c in zip(best['names'], best['coef']):
    print(f'  {nm:5s} = {c:+.6f}')

fig, (axp, axr) = plt.subplots(1, 2, figsize=(12, 4))
axp.scatter(err, best['pred'])
axp.plot([err.min(), err.max()], [err.min(), err.max()], 'k--')
axp.set_xlabel('measured error'); axp.set_ylabel('predicted error')
axp.set_title('predicted vs measured')
axr.scatter(cmd, err - best['pred'], c=speed)
axr.axhline(0, color='k', lw=0.5)
axr.set_xlabel('commanded angle'); axr.set_ylabel('residual (deg)')
axr.set_title('residuals')
for ax in (axp, axr):
    ax.grid(alpha=0.3)

In [ ]:
# Direction asymmetry check (CCW vs CW)
for sign, lbl in ((+1, 'CCW'), (-1, 'CW')):
    m = np.sign(cmd) == sign
    if not m.any():
        continue
    slope = np.polyfit(np.abs(cmd[m]), np.abs(err[m]), 1)[0]
    print(f'{lbl}: mean |err|={np.abs(err[m]).mean():.1f} deg   |err| per deg slope={slope:+.4f}')

## Applying the model

The fit gives `error(cmd, speed)`. To turn a desired `target` at a given `speed`,
command `cmd` such that `cmd + error(cmd, speed) ≈ target`. Since `error ≪ cmd`, one
step is plenty:

```
cmd ≈ target − error(target, speed)
```

Store the chosen coefficients in the robot config (e.g. a `rotation_model` block:
the term list + coefficients) and have the turn command evaluate `error(target,
speed)` and pre-subtract it. Re-run `rotation_sweep.py` to append more data and
re-fit.